# Feature Engineering

## Objective

The objective of this notebook is to transform the cleaned dataset into a machine learning-ready format. This includes encoding categorical variables, scaling numerical features, splitting the dataset into training and testing sets, and preparing the data for model training.

In [57]:
# Import required libraries

import pandas as pd
import numpy as np

# Import encoding and preprocessing tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder , OrdinalEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

## Loading the Cleaned Dataset

The cleaned dataset is loaded for feature engineering. This dataset will be transformed into a machine learning-ready format.

In [58]:
# Load the cleaned dataset

df = pd.read_csv("../Data/02_Cleaned/Student Social Media And Mental Health Impact Cleaned Data.csv")


## Dataset Overview

In [59]:
# Display the first five rows
df.head()

,Age,Gender,Country,Academic_Level,Most_Used_Platform,Purpose_Of_Use,Avg_Daily_Usage_Hours,Daily_Unlocks,Study_Hours,Physical_Activity_Hours,Sleep_Hours_Per_Night,Stress_Level,Mental_Health_Score
0,21,Male,Other,Undergraduate,Facebook,Networking,4.0,134,4.5,2.2,6.7,Medium,6.8
1,23,Female,Other,Graduate,LinkedIn,Education,1.6,73,7.0,2.4,8.6,Low,7.6
2,22,Male,Canada,Undergraduate,Instagram,Entertainment,4.6,166,4.0,1.8,6.7,Medium,7.0
3,18,Male,Other,High School,Snapchat,Entertainment,7.0,220,1.0,1.7,5.4,Very High,5.3
4,24,Female,Other,Graduate,Facebook,Networking,7.5,237,1.0,1.1,5.0,Very High,4.4


In [60]:
# Display the shape of the dataset
df.shape

(4998, 13)

In [61]:
# Diplay the data types of each column
df.dtypes

Age                          int64
Gender                         str
Country                        str
Academic_Level                 str
Most_Used_Platform             str
Purpose_Of_Use                 str
Avg_Daily_Usage_Hours      float64
Daily_Unlocks                int64
Study_Hours                float64
Physical_Activity_Hours    float64
Sleep_Hours_Per_Night      float64
Stress_Level                   str
Mental_Health_Score        float64
dtype: object

## Separating Features and Target Variable

The input features (X) and the target variable (y) are separated before applying feature engineering techniques.

In [62]:
# Separate input features and target variable

X = df.drop("Mental_Health_Score", axis=1) #Take everything except the Mental Health Score.

y = df["Mental_Health_Score"] # Target Variable 

# Display the shape of features and target
print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

Features Shape: (4998, 12)
Target Shape: (4998,)


### Why Features and Target are Separated?

Machine learning models learn the relationship between input features (X) and the target variable (y). Separating them ensures that the model predicts the target using only the input features without accessing the actual target values during training.

## Identifying Categorical and Numerical Features

Categorical and numerical features are identified to apply appropriate preprocessing techniques such as encoding and scaling.

In [63]:
# Identify categorical and numerical features

categorical_features = X.select_dtypes(include=['object', 'string']).columns.tolist()
numerical_features = X.select_dtypes(include=['number']).columns.tolist()

print("Categorical Features:")
print(categorical_features)

print("\nNumerical Features:")
print(numerical_features)

Categorical Features:
['Gender', 'Country', 'Academic_Level', 'Most_Used_Platform', 'Purpose_Of_Use', 'Stress_Level']

Numerical Features:
['Age', 'Avg_Daily_Usage_Hours', 'Daily_Unlocks', 'Study_Hours', 'Physical_Activity_Hours', 'Sleep_Hours_Per_Night']


### Key Insights

- The dataset contains 6 categorical features and 6 numerical features.
- Categorical features represent student characteristics, preferences, and stress level.
- Numerical features represent age, social-media usage, study, physical activity, and sleep patterns.
- `Mental_Health_Score` is the target variable and is excluded from the input features.

## Categorical Feature Analysis

In [64]:
# Check unique values in each categorical feature

for column in categorical_features:
    print(f"\n{column}:")
    print(df[column].unique())


Gender:
<ArrowStringArray>
['Male', 'Female']
Length: 2, dtype: str

Country:
<ArrowStringArray>
[      'Other',      'Canada',         'USA',       'India',   'Australia',
          'UK',     'Germany',  'Bangladesh',      'Brazil',       'Japan',
 ...
     'Lebanon',        'Iraq',       'Yemen',       'Syria', 'Afghanistan',
    'Pakistan',       'Nepal',      'Bhutan',   'Sri Lanka',    'Maldives']
Length: 111, dtype: str

Academic_Level:
<ArrowStringArray>
['Undergraduate', 'Graduate', 'High School']
Length: 3, dtype: str

Most_Used_Platform:
<ArrowStringArray>
[ 'Facebook',  'LinkedIn', 'Instagram',  'Snapchat',   'Twitter',   'YouTube',
    'TikTok',      'LINE', 'KakaoTalk', 'VKontakte',  'WhatsApp',    'WeChat']
Length: 12, dtype: str

Purpose_Of_Use:
<ArrowStringArray>
['Networking', 'Education', 'Entertainment', 'News']
Length: 4, dtype: str

Stress_Level:
<ArrowStringArray>
['Medium', 'Low', 'Very High', 'High']
Length: 4, dtype: str


## Encoding Plan

| Feature | Categories | Encoding |
|---|---:|---|
| `Gender` | 2 | One-Hot |
| `Country` | 111 | One-Hot* |
| `Academic_Level` | 3 | One-Hot |
| `Most_Used_Platform` | 12 | One-Hot |
| `Purpose_Of_Use` | 4 | One-Hot |
| `Stress_Level` | 4 | Ordinal* |

### Why These Encoding Methods?

- **One-Hot Encoding:** Used for categories that do not have a natural order. It creates separate binary columns for each category.
- **Ordinal Encoding:** Used for `Stress_Level` because its categories have a meaningful order:
  `Low < Medium < High < Very High`.
- `Stress_Level` is therefore represented as:
  `Low → 0`, `Medium → 1`, `High → 2`, `Very High → 3`.

### ⚠️ Country Feature

- `Country` contains **111 categories**.
- One-Hot Encoding would create 111 additional columns.
- Before encoding, country frequencies will be checked to identify very rare categories.
- Based on the frequency distribution, we will decide whether to keep all countries or group rare countries.

## Categorical Feature Distribution

In [65]:
df["Country"].value_counts()

Country
Other         1879
India          389
USA            354
Canada         230
Australia      198
              ... 
Montenegro       1
Bosnia           1
Kuwait           1
Yemen            1
Syria            1
Name: count, Length: 111, dtype: int64

### Key Insights

- `Country` contains 111 categories.
- The distribution is highly uneven, with some countries having very few records.
- Several countries have only one observation.
- One-Hot Encoding all 111 categories may create unnecessary sparse features.
- Rare countries can be grouped into a separate `Rare_Country` category before encoding.

### check how many countries are rare?

In [66]:
# Count countries with fewer than 5 observations
country_counts = df["Country"].value_counts()

rare_countries = country_counts[country_counts < 5]

print("Number of rare countries:", len(rare_countries))
print("\nRare countries:")
print(rare_countries)

Number of rare countries: 64

Rare countries:
Country
Belgium         4
Greece          4
South Africa    4
Peru            4
Panama          4
               ..
Montenegro      1
Bosnia          1
Kuwait          1
Yemen           1
Syria           1
Name: count, Length: 64, dtype: int64


### Rare Category Analysis

- The `Country` feature contains 111 categories.
- 64 countries have fewer than 5 observations.
- Several countries have only 1–4 observations.
- These very rare categories may provide limited information to the model.
- Countries with fewer than 5 observations will be grouped into `Rare_Country`.

## Handling Rare Country Categories

- Country count >= 5  → Keep original country
- Country count < 5   → Rare_Country

In [67]:
# Count the frequency of each country
country_counts = df["Country"].value_counts()

# Create a processed country feature
# Countries with fewer than 5 observations are grouped as Rare_Country
df["Country_Processed"] = df["Country"].apply(
    lambda x: "Rare_Country" if country_counts[x] < 5 else x
)

In [68]:
# Compare original and processed country categories
print("Original categories:", df["Country"].nunique())
print("Processed categories:", df["Country_Processed"].nunique())

Original categories: 111
Processed categories: 48


In [69]:
# Recreate X after adding Country_Processed
X = df.drop(columns=["Mental_Health_Score"])

# Remove the original Country feature
# Country_Processed will be used instead
X = X.drop(columns=["Country"])

# Check the updated feature columns
X.columns.tolist()

['Age',
 'Gender',
 'Academic_Level',
 'Most_Used_Platform',
 'Purpose_Of_Use',
 'Avg_Daily_Usage_Hours',
 'Daily_Unlocks',
 'Study_Hours',
 'Physical_Activity_Hours',
 'Sleep_Hours_Per_Night',
 'Stress_Level',
 'Country_Processed']

### Key Insights

- The original `Country` feature contained 111 categories.
- 64 rare countries were identified using a frequency threshold of fewer than 5 observations.
- Rare countries were grouped into `Rare_Country`.
- The processed feature now contains 48 categories.
- The original `Country` feature was preserved for reference.
- This reduces sparse categories before One-Hot Encoding.

## Final Categorical Features

- `Country_Processed` is used instead of the original `Country` feature.
- `Country_Processed` contains the grouped rare-country categories.
- `Country_Processed` will be used for encoding to reduce the number of sparse country categories.
- All other categorical features are retained without modification.
- The original `Country` feature is kept unchanged for reference.
- The final categorical features are:

  - `Gender`
  - `Country_Processed`
  - `Academic_Level`
  - `Most_Used_Platform`
  - `Purpose_Of_Use`
  - `Stress_Level`





## Categorical Feature Encoding

- One-Hot Encoding is used for non-ordered categorical features.
- Ordinal Encoding is used for `Stress_Level` because its categories have a natural order.
- `Country_Processed` is used instead of the original `Country` feature.

In [70]:
# Import encoding and preprocessing tools
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

In [71]:
# Define categorical features for encoding
one_hot_features = [
    "Gender",
    "Country_Processed",
    "Academic_Level",
    "Most_Used_Platform",
    "Purpose_Of_Use"
]

ordinal_features = ["Stress_Level"]

In [72]:
# Define the correct order for stress levels
stress_order = [
    ["Low", "Medium", "High", "Very High"]
]

In [73]:
# Create preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            one_hot_features
        ),
        (
            "ordinal",
            OrdinalEncoder(categories=stress_order),
            ordinal_features
        )
    ],
    remainder="passthrough"
)

### Key Insights

- Five categorical features are converted using One-Hot Encoding.
- `Stress_Level` is converted using Ordinal Encoding.
- The stress order is preserved as `Low < Medium < High < Very High`.
- `handle_unknown="ignore"` allows the model to handle unseen categories during prediction.
- Numerical features are kept unchanged at this stage.

## Apply Categorical Encoding

In [74]:
# Apply the encoding transformations to the feature data
X_encoded = preprocessor.fit_transform(X)

# Check the shape of the encoded data
X_encoded.shape

(4998, 76)

### Key Insights

- The dataset contains 4,998 student records.
- After categorical encoding, the number of features increased from 12 to 76.
- This increase is mainly due to One-Hot Encoding of categorical variables.
- All categorical features are now represented in numerical form.
- The dataset is now ready for numerical feature scaling.

## Numerical Feature Scaling

- Numerical features have different ranges and units.
- Features with larger values can have a greater influence on some ML algorithms.
- Scaling brings numerical features to a comparable range.
- `StandardScaler` is used to standardize the numerical features.

#### In simple words:
- Our numerical features have very different values.
- For example, `Age` is around 18–30, while `Daily_Unlocks` can be above 200.
- Because of these different ranges, some features may have more influence than others.
- Scaling makes these values more balanced for the machine learning model.
- `StandardScaler` is used to bring the numerical features to a similar scale.

Scaling = bringing numerical features to a similar scale so the model can work with them fairly.

In [75]:
# Numerical features that need scaling
numerical_features = [
    "Age",
    "Avg_Daily_Usage_Hours",
    "Daily_Unlocks",
    "Study_Hours",
    "Physical_Activity_Hours",
    "Sleep_Hours_Per_Night"
]

print("Numerical features to scale:")
print(numerical_features)

Numerical features to scale:
['Age', 'Avg_Daily_Usage_Hours', 'Daily_Unlocks', 'Study_Hours', 'Physical_Activity_Hours', 'Sleep_Hours_Per_Night']


In [76]:
# Import StandardScaler for numerical feature scaling
from sklearn.preprocessing import StandardScaler

### Why StandardScaler?

- StandardScaler transforms numerical features so that they have a mean close to 0 and a standard deviation close to 1.
- It is useful when numerical features have different scales.
- Example: `Daily_Unlocks` has much larger values than `Age`.

In [77]:
# Create the scaler for numerical features
scaler = StandardScaler()

In [78]:
# ReCreate the complete preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            one_hot_features
        ),
        (
            "ordinal",
            OrdinalEncoder(categories=stress_order),
            ordinal_features
        ),
        (
            "scaler",
            StandardScaler(),
            numerical_features
        )
    ],
    remainder="drop"
)

In [79]:
# Apply encoding and scaling to the feature data
X_processed = preprocessor.fit_transform(X)

# Check the shape of the processed data
X_processed.shape

(4998, 76)

### Key Insights

- The dataset contains **4,998 student records**.
- After encoding and scaling, the feature set contains **76 features**.
- Categorical features have been converted into numerical values.
- Numerical features have been standardized using `StandardScaler`.
- The processed features are now ready for the next stage of the machine learning workflow.

In [80]:
# Get the names of all processed features
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))
print(feature_names)

Number of processed features: 76
['onehot__Gender_Female' 'onehot__Gender_Male'
 'onehot__Country_Processed_Afghanistan'
 'onehot__Country_Processed_Andorra' 'onehot__Country_Processed_Argentina'
 'onehot__Country_Processed_Australia'
 'onehot__Country_Processed_Bangladesh' 'onehot__Country_Processed_Brazil'
 'onehot__Country_Processed_Canada' 'onehot__Country_Processed_China'
 'onehot__Country_Processed_Costa Rica' 'onehot__Country_Processed_Cyprus'
 'onehot__Country_Processed_Denmark' 'onehot__Country_Processed_Egypt'
 'onehot__Country_Processed_Finland' 'onehot__Country_Processed_France'
 'onehot__Country_Processed_Georgia' 'onehot__Country_Processed_Germany'
 'onehot__Country_Processed_India' 'onehot__Country_Processed_Ireland'
 'onehot__Country_Processed_Italy' 'onehot__Country_Processed_Japan'
 'onehot__Country_Processed_Malaysia' 'onehot__Country_Processed_Maldives'
 'onehot__Country_Processed_Mexico' 'onehot__Country_Processed_Moldova'
 'onehot__Country_Processed_Nepal' 'onehot

In [81]:
# Convert the processed array into a DataFrame
X_processed_df = pd.DataFrame(
    X_processed,
    columns=feature_names,
    index=X.index
)

# Display the first 5 rows
X_processed_df.head()

,onehot__Gender_Female,onehot__Gender_Male,onehot__Country_Processed_Afghanistan,onehot__Country_Processed_Andorra,onehot__Country_Processed_Argentina,onehot__Country_Processed_Australia,onehot__Country_Processed_Bangladesh,onehot__Country_Processed_Brazil,onehot__Country_Processed_Canada,onehot__Country_Processed_China,...,onehot__Purpose_Of_Use_Entertainment,onehot__Purpose_Of_Use_Networking,onehot__Purpose_Of_Use_News,ordinal__Stress_Level,scaler__Age,scaler__Avg_Daily_Usage_Hours,scaler__Daily_Unlocks,scaler__Study_Hours,scaler__Physical_Activity_Hours,scaler__Sleep_Hours_Per_Night
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,0.102425,-0.652077,-0.873996,0.911362,0.672706,0.053499
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.254100,-2.103165,-2.297383,2.438856,0.972460,1.609042
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.678263,-0.289305,-0.127302,0.605863,0.073199,0.053499
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,3.0,-1.625088,1.161782,1.132745,-1.227130,-0.076678,-1.010819
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,3.0,1.829938,1.464092,1.529427,-1.227130,-0.975938,-1.338302


In [82]:
# Check the shape of the processed feature DataFrame
print("Processed Features Shape:", X_processed_df.shape)

Processed Features Shape: (4998, 76)


In [83]:
# Add the target variable back to the processed dataset
processed_df = X_processed_df.copy()
processed_df["Mental_Health_Score"] = df["Mental_Health_Score"].values

# Check the final dataset shape
print("Final Processed Dataset Shape:", processed_df.shape)

Final Processed Dataset Shape: (4998, 77)


### Key Insights

- The 76 processed features were converted into a structured DataFrame.
- The original target `Mental_Health_Score` was added back to the dataset.
- The final processed dataset contains **4,998 rows and 77 columns**.
- The 76 columns are input features and `Mental_Health_Score` is the target variable.

### Before saving — one quick check

In [84]:
# Check for missing values
print("Total missing values:", processed_df.isnull().sum().sum())

# Check for duplicate rows
print("Duplicate rows:", processed_df.duplicated().sum())

# Check final data types
processed_df.dtypes.value_counts()

Total missing values: 0
Duplicate rows: 0


float64    77
Name: count, dtype: int64

In [85]:
# Preview the final processed dataset
processed_df.head()

,onehot__Gender_Female,onehot__Gender_Male,onehot__Country_Processed_Afghanistan,onehot__Country_Processed_Andorra,onehot__Country_Processed_Argentina,onehot__Country_Processed_Australia,onehot__Country_Processed_Bangladesh,onehot__Country_Processed_Brazil,onehot__Country_Processed_Canada,onehot__Country_Processed_China,...,onehot__Purpose_Of_Use_Networking,onehot__Purpose_Of_Use_News,ordinal__Stress_Level,scaler__Age,scaler__Avg_Daily_Usage_Hours,scaler__Daily_Unlocks,scaler__Study_Hours,scaler__Physical_Activity_Hours,scaler__Sleep_Hours_Per_Night,Mental_Health_Score
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,1.0,0.102425,-0.652077,-0.873996,0.911362,0.672706,0.053499,6.8
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.254100,-2.103165,-2.297383,2.438856,0.972460,1.609042,7.6
2,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.678263,-0.289305,-0.127302,0.605863,0.073199,0.053499,7.0
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,3.0,-1.625088,1.161782,1.132745,-1.227130,-0.076678,-1.010819,5.3
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,3.0,1.829938,1.464092,1.529427,-1.227130,-0.975938,-1.338302,4.4


### Final Preprocessing Insights

- No missing values are present.
- No duplicate records are present.
- All 76 input features are numerical after preprocessing.
- `Mental_Health_Score` -- Target variable is also stored as a numerical value.
- The final processed dataset contains 4,998 records and 77 columns.
- The dataset is ready to be saved for model training.

## Save the Processed Dataset

In [87]:
# Save the final processed dataset
processed_df.to_csv(
    "../Data/03_Processed/mental_health_processed.csv",
    index=False
)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!


In [88]:
# Check that the processed file was saved correctly
import os

file_path = "../Data/03_Processed/mental_health_processed.csv"

print("File exists:", os.path.exists(file_path))

File exists: True
